# Harvard Dataverse - U.S. House Elections 1976-2024

## Data Cleaning & Preparation

Source: Harvard Dataverse (https://dataverse.harvard.edu)
File: 1976-2024-house.tab

**Steps performed:**
1. Loaded tab-separated file and confirmed column structure
2. Filtered for 2024 general elections only
3. Dropped write-in candidates
4. Standardized party column (REPUBLICAN, DEMOCRAT, OTHER)
5. Handled fusion tickets by keeping highest vote count per candidate per district

**Output:** house_2024_clean.csv

In [1]:
import pandas as pd
import os

df = pd.read_csv('j_notebooks/raw_source/1976-2024-house.tab', sep=',', low_memory = False)
df.columns

Index(['year', 'state', 'state_po', 'state_fips', 'state_cen', 'state_ic',
       'office', 'district', 'stage', 'runoff', 'special', 'candidate',
       'party', 'writein', 'mode', 'candidatevotes', 'totalvotes',
       'unofficial', 'version', 'fusion_ticket'],
      dtype='object')

In [2]:
# filter for 2024 and general election only
df_2024 = df[(df['year'] == 2024) & (df['stage'] == 'GEN')]
df_2024.shape

(1353, 20)

In [3]:
df_2024.head(10)

,year,state,state_po,state_fips,state_cen,state_ic,office,district,stage,runoff,special,candidate,party,writein,mode,candidatevotes,totalvotes,unofficial,version,fusion_ticket
32452,2024,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,BARRY MOORE,REPUBLICAN,False,TOTAL,258619,329854,False,20250910,False
32453,2024,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,TOM HOLMES,DEMOCRAT,False,TOTAL,70929,329854,False,20250910,False
32454,2024,ALABAMA,AL,1,63,41,US HOUSE,1,GEN,False,False,WRITEIN,NaN,True,TOTAL,306,329854,False,20250910,False
32455,2024,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,False,False,CAROLEENE DOBSON,REPUBLICAN,False,TOTAL,131414,289674,False,20250910,False
32456,2024,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,False,False,SHOMARI FIGURES,DEMOCRAT,False,TOTAL,158041,289674,False,20250910,False
32457,2024,ALABAMA,AL,1,63,41,US HOUSE,2,GEN,False,False,WRITEIN,NaN,True,TOTAL,219,289674,False,20250910,False
32458,2024,ALABAMA,AL,1,63,41,US HOUSE,3,GEN,False,False,MIKE ROGERS,REPUBLICAN,False,TOTAL,243848,249008,False,20250910,False
32459,2024,ALABAMA,AL,1,63,41,US HOUSE,3,GEN,False,False,WRITEIN,NaN,True,TOTAL,5160,249008,False,20250910,False
32460,2024,ALABAMA,AL,1,63,41,US HOUSE,4,GEN,False,False,ROBERT B. ADERHOLT,REPUBLICAN,False,TOTAL,274498,277872,False,20250910,False
32461,2024,ALABAMA,AL,1,63,41,US HOUSE,4,GEN,False,False,WRITEIN,NaN,True,TOTAL,3374,277872,False,20250910,False


In [4]:
# checking # of states
df_2024['state'].nunique()

51

In [5]:
# checking unique values in party column
df_2024['party'].value_counts()

party
REPUBLICAN                          427
DEMOCRAT                            422
LIBERTARIAN                          72
INDEPENDENT                          37
GREEN                                30
CONSERVATIVE                         22
WORKING FAMILIES                     12
UNAFFILIATED                         10
CONSTITUTION                          9
WORKING CLASS                         8
UNITY PARTY                           6
DEMOCRATIC-FARMER-LABOR               5
APPROVAL VOTING PARTY                 5
U.S. TAXPAYERS                        4
COMMON SENSE                          3
PACIFIC GREEN                         3
NO POLITICAL PARTY                    3
INDEPENDENT AMERICAN                  3
NO PARTY                              2
ALLIANCE                              2
NO PARTY AFFILIATION                  2
LAROUCHE                              1
COMMON SENSE SUFFOLK                  1
FORWARD PARTY                         1
AMERICAN CONSTITUTION             

In [6]:
#checking for missing values
df_2024.isnull().sum()

year                0
state               0
state_po            0
state_fips          0
state_cen           0
state_ic            0
office              0
district            0
stage               0
runoff              0
special             0
candidate           0
party             236
writein             0
mode                0
candidatevotes      0
totalvotes          0
unofficial          0
version             0
fusion_ticket       0
dtype: int64

In [7]:
# checking basic stats on votes
df_2024['candidatevotes'].describe()

count      1353.000000
mean     110480.584627
std       94247.466510
min           1.000000
25%        7795.000000
50%      112018.000000
75%      191363.000000
max      352286.000000
Name: candidatevotes, dtype: float64

In [8]:
# checking write-in candidates connection to nulls
df_2024[df_2024['party'].isnull()]['writein'].value_counts()

writein
False    129
True     107
Name: count, dtype: int64

In [9]:
# checking fusion tickets
df_2024['fusion_ticket'].value_counts()

fusion_ticket
False    1316
True       37
Name: count, dtype: int64

In [10]:
# 
df_2024[
   (df_2024['party'].isnull()) & 
   (df_2024['writein'] == False)
][['candidate', 'state', 'district', 'party']].head(20)

,candidate,state,district,party
32661,WORKING FAMILIES,CONNECTICUT,1,NaN
32666,INDEPENDENT,CONNECTICUT,3,NaN
32678,WORKING FAMILIES,CONNECTICUT,5,NaN
32685,OVER VOTES,DISTRICT OF COLUMBIA,0,NaN
32686,UNDER VOTES,DISTRICT OF COLUMBIA,0,NaN
32782,BLANK VOTES,HAWAII,1,NaN
32784,OVER VOTES,HAWAII,1,NaN
32787,BLANK VOTES,HAWAII,2,NaN
32789,OVER VOTES,HAWAII,2,NaN
32797,OVER VOTES,IDAHO,1,NaN


In [11]:
# dropping write-in candidates
df_2024 = df_2024[df_2024['writein'] == False]

df_2024.shape

(1234, 20)

In [12]:
# labeling null party as OTHER
df_2024['party'] = df_2024['party'].fillna('OTHER')

df_2024['party'].isnull().sum()

0

In [13]:
# Keeping REPUBLICAN and DEMOCRAT party as is, everything else becomes OTHER
df_2024['party'] = df_2024['party'].apply(
   lambda x: x if x in ['REPUBLICAN', 'DEMOCRAT'] else 'OTHER'
)

df_2024['party'].value_counts()

party
REPUBLICAN    427
DEMOCRAT      420
OTHER         387
Name: count, dtype: int64

## Handling Fusion Tickets

Some candidates appear multiple times under different party labels in the same 
district (common iYorkJseyTc). 

To avoid duplicates when merging with FEC data laI'mr, weing keep only the row 
with the highest vote count per candidate per ditrict.

In [14]:
# Keeping the row with the most votes per candidate per district
df_2024 = df_2024.sort_values('candidatevotes', ascending=False)
df_2024 = df_2024.drop_duplicates(
   subset=['candidate', 'state', 'district'], 
   keep='first'
)

df_2024.shape

(1199, 20)

In [15]:
df_2024.to_csv('j_notebooks/clean_data/house_2024_clean.csv', index=False)

print('File exported!')

File exported!
